In [ ]:
# Instrument dataset + MERT COMPLETE PIPELINE
import os
import glob
import numpy as np
import pandas as pd
import torch
import torchaudio
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    Wav2Vec2FeatureExtractor
)

import kagglehub

# Download latest version
path = kagglehub.dataset_download("abdulvahap/music-instrunment-sounds-for-classification")

print("Dataset path:")
print(path)


DATA_DIR = os.path.join(
    path,
    "Data"
)


OUTPUT_3S = os.path.join(
    DATA_DIR,
    "features_3_sec_MERT.csv"
)

ERROR_LOG = os.path.join(
    DATA_DIR,
    "MERT_audio_errors.csv"
)

print("\nDATA_DIR :", DATA_DIR)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nUsing device:", device)

MODEL_NAME = "m-a-p/MERT-v1-95M"

processor = Wav2Vec2FeatureExtractor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

TARGET_SR = processor.sampling_rate

print("MERT sample rate:", TARGET_SR)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = model.to(device)
model.eval()

print(
    "MERT hidden size:",
    model.config.hidden_size
)

assert model.config.hidden_size == 768
assert TARGET_SR == 24000

print("MERT loaded successfully.")

100%|██████████| 4.59G/4.59G [03:37<00:00, 22.7MB/s]

Extracting files...


Dataset path:
/root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1

DATA_DIR : /root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1/Data

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

MERT sample rate: 24000


...

[transformers] A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- modeling_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

MERT hidden size: 768
MERT loaded successfully.


In [ ]:
DATA_DIR

'/root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1/Data'

In [ ]:
import os

def print_directory_tree(startpath, max_depth=3):
    """Prints a visual tree structure of the directory."""
    print(f"📦 Root Directory: {startpath}\n")

    start_level = startpath.rstrip(os.sep).count(os.sep)
    for root, dirs, files in os.walk(startpath):
        level = root.rstrip(os.sep).count(os.sep) - start_level
        if level >= max_depth:
            continue

        indent = '   ' * level
        print(f"{indent}📁 {os.path.basename(root)}/")

        # Print first few files to avoid cluttering the screen
        sub_indent = '   ' * (level + 1)
        for f in files[:5]:
            print(f"{sub_indent}📄 {f}")
        if len(files) > 5:
            print(f"{sub_indent}📄 ... and {len(files) - 5} more files")

# Pass your 'path' variable directly here
print_directory_tree(path, max_depth=3)


📦 Root Directory: /root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1

📁 1/
   📁 music_dataset/
      📁 Harmonium/
         📄 1129.wav
         📄 803.wav
         📄 313.wav
         📄 714.wav
         📄 66.wav
         📄 ... and 1309 more files
      📁 Floor_Tom/
         📄 313.wav
         📄 66.wav
         📄 203.wav
         📄 28.wav
         📄 54.wav
         📄 ... and 401 more files
      📁 Harmonica/
         📄 66.wav
         📄 28.wav
         📄 54.wav
         📄 93.wav
         📄 123.wav
         📄 ... and 126 more files
      📁 Piano/
         📄 313.wav
         📄 66.wav
         📄 203.wav
         📄 28.wav
         📄 54.wav
         📄 ... and 570 more files
      📁 Acoustic_Guitar/
         📄 1129.wav
         📄 2647.wav
         📄 1849.wav
         📄 803.wav
         📄 3081.wav
         📄 ... and 3649 more files
      📁 Hi_Hats/
         📄 313.wav
         📄 66.wav
         📄 203.wav
         📄 28.wav
         📄 54.wav
         📄 

In [ ]:
# ============================================================
# MERT FEATURE EXTRACTION
# 3-second clips
# 1000 randomly selected samples per eligible class
# MERT-v1-95M
# Layer 6 + mean pooling
# ============================================================

import os
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torchaudio
from tqdm.auto import tqdm


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# -----------------------------
# Experiment parameters
# -----------------------------

CLIP_SECONDS = 3
CLIP_SAMPLES = TARGET_SR * CLIP_SECONDS

SAMPLES_PER_CLASS = 1000

# Recommended MERT layer
MERT_LAYER = 6

# Batch size
# Start with 8 or 16 if GPU memory is limited.
BATCH_SIZE = 16


# ============================================================
# 2. DATASET PATH
# ============================================================

# Your actual structure is:
#
# /root/.cache/kagglehub/datasets/
#     abdulvahap/
#         music-instrunment-sounds-for-classification/
#             versions/
#                 1/
#                     music_dataset/
#                         Accordion/
#                         Acoustic_Guitar/
#                         ...
#
# Therefore:
#
DATASET_ROOT = os.path.join(
    path,
    "music_dataset"
)

print("Dataset root:")
print(DATASET_ROOT)


# ============================================================
# 3. ELIGIBLE CLASSES
# ============================================================

# Classes with > 1000 original samples

ELIGIBLE_CLASSES = [
    "Accordion",
    "Acoustic_Guitar",
    "Banjo",
    "Bass_Guitar",
    "Drum_set",
    "Electro_Guitar",
    "flute",
    "Harmonium",
    "Keyboard",
    "Mandolin",
    "Organ",
    "Shakers",
    "Trombone",
    "Horn",
]


# ============================================================
# 4. CHECK CLASSES
# ============================================================

print("\nChecking dataset classes...\n")

for class_name in ELIGIBLE_CLASSES:

    class_dir = os.path.join(
        DATASET_ROOT,
        class_name
    )

    if not os.path.isdir(class_dir):
        print(
            f"WARNING: Class directory not found: "
            f"{class_dir}"
        )

    else:
        wav_count = len(
            glob.glob(
                os.path.join(
                    class_dir,
                    "*.wav"
                )
            )
        )

        print(
            f"{class_name:20s} : "
            f"{wav_count} wav files"
        )


# ============================================================
# 5. RANDOMLY SELECT 1000 FILES PER CLASS
# ============================================================

selected_files = []

print("\nSelecting files...\n")

for class_name in ELIGIBLE_CLASSES:

    class_dir = os.path.join(
        DATASET_ROOT,
        class_name
    )

    files = glob.glob(
        os.path.join(
            class_dir,
            "*.wav"
        )
    )

    # Safety check
    if len(files) < SAMPLES_PER_CLASS:

        raise ValueError(
            f"{class_name} has only {len(files)} files. "
            f"Need at least {SAMPLES_PER_CLASS}."
        )

    # Sort before random sampling so that
    # selection is reproducible
    files = sorted(files)

    rng = random.Random(
        SEED + hash(class_name) % 100000
    )

    selected = rng.sample(
        files,
        SAMPLES_PER_CLASS
    )

    for file_path in selected:

        selected_files.append({
            "filepath": file_path,
            "label": class_name
        })


print(
    "\nTotal selected files:",
    len(selected_files)
)

Dataset root:
/root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1/music_dataset

Checking dataset classes...

Accordion            : 3581 wav files
Acoustic_Guitar      : 3654 wav files
Banjo                : 2998 wav files
Bass_Guitar          : 3613 wav files
Drum_set             : 3648 wav files
Electro_Guitar       : 1316 wav files
flute                : 3719 wav files
Harmonium            : 1314 wav files
Keyboard             : 2041 wav files
Mandolin             : 2458 wav files
Organ                : 1442 wav files
Shakers              : 1357 wav files
Trombone             : 2965 wav files
Horn                 : 1258 wav files

Selecting files...


Total selected files: 14000


In [ ]:
# ============================================================
# 6. VERIFY CLASS BALANCE
# ============================================================

selection_df = pd.DataFrame(
    selected_files
)

print("\nSelected samples per class:\n")

print(
    selection_df["label"]
    .value_counts()
    .sort_index()
)

assert len(selection_df) == (
    len(ELIGIBLE_CLASSES)
    * SAMPLES_PER_CLASS
)

assert (
    selection_df["label"]
    .value_counts()
    == SAMPLES_PER_CLASS
).all()


# ============================================================
# 7. OUTPUT PATHS
# ============================================================

OUTPUT_3S = os.path.join(
    DATASET_ROOT,
    "features_3_sec_MERT_layer6.csv"
)

ERROR_LOG = os.path.join(
    DATASET_ROOT,
    "MERT_audio_errors.csv"
)

print("\nFeature output:")
print(OUTPUT_3S)

print("\nError log:")
print(ERROR_LOG)


Selected samples per class:

label
Accordion          1000
Acoustic_Guitar    1000
Banjo              1000
Bass_Guitar        1000
Drum_set           1000
Electro_Guitar     1000
Harmonium          1000
Horn               1000
Keyboard           1000
Mandolin           1000
Organ              1000
Shakers            1000
Trombone           1000
flute              1000
Name: count, dtype: int64

Feature output:
/root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1/music_dataset/features_3_sec_MERT_layer6.csv

Error log:
/root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1/music_dataset/MERT_audio_errors.csv


In [ ]:
# file_path = "/root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1/music_dataset/MERT_audio_errors.csv"

# try:
#     with open(file_path, 'r', encoding='utf-8') as f:
#         # Reads and prints the first 50 lines to prevent overwhelming the screen
#         for i in range(50):
#             line = f.readline()
#             if not line:
#                 break
#             print(line.strip())
# except FileNotFoundError:
#     print(f"❌ Error: The file at {file_path} was not found.")
# except Exception as e:
#     print(f"❌ Could not read the file: {e}")

❌ Error: The file at /root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1/music_dataset/MERT_audio_errors.csv was not found.


In [ ]:
# ============================================================
# 8. AUDIO PREPROCESSING FUNCTION
# ============================================================

def load_and_prepare_audio(
    file_path,
    target_sr=TARGET_SR,
    clip_samples=CLIP_SAMPLES
):
    """
    Load audio, convert to mono, resample to 24 kHz,
    and make it exactly 3 seconds long.

    If longer than 3 sec:
        take the first 3 sec.

    If shorter than 3 sec:
        zero-pad at the end.
    """

    waveform, sample_rate = torchaudio.load(
        file_path
    )

    # -----------------------------------------
    # Convert stereo/multichannel -> mono
    # -----------------------------------------

    if waveform.ndim == 2:

        if waveform.shape[0] > 1:
            waveform = waveform.mean(
                dim=0,
                keepdim=True
            )

    # -----------------------------------------
    # Resample
    # -----------------------------------------

    if sample_rate != target_sr:

        waveform = torchaudio.functional.resample(
            waveform,
            orig_freq=sample_rate,
            new_freq=target_sr
        )

    # -----------------------------------------
    # Remove channel dimension
    # -----------------------------------------

    waveform = waveform.squeeze(0)

    # -----------------------------------------
    # Make exactly 3 seconds
    # -----------------------------------------

    current_samples = waveform.shape[0]

    if current_samples >= clip_samples:

        # Take first 3 seconds
        waveform = waveform[:clip_samples]

    else:

        # Zero pad
        padding = clip_samples - current_samples

        waveform = torch.nn.functional.pad(
            waveform,
            (0, padding)
        )

    # -----------------------------------------
    # Safety check
    # -----------------------------------------

    assert waveform.shape[0] == clip_samples

    return waveform

In [ ]:
# ============================================================
# MERT LAYER 6 EXTRACTION
# ============================================================

MERT_LAYER = 6                 # 1-based layer number
MERT_LAYER_INDEX = MERT_LAYER - 1

captured_layer_output = None


def capture_layer_output(module, inputs, output):
    """
    Capture the output of the selected MERT Transformer layer.
    """
    global captured_layer_output

    # HubertEncoderLayer returns a tuple.
    if isinstance(output, tuple):
        captured_layer_output = output[0]
    else:
        captured_layer_output = output


# ------------------------------------------------------------
# Register hook on Transformer block 6
# ------------------------------------------------------------

layer_handle = model.encoder.layers[
    MERT_LAYER_INDEX
].register_forward_hook(
    capture_layer_output
)


print(
    f"MERT feature extraction layer: "
    f"{MERT_LAYER}"
)

print(
    f"Internal layer index: "
    f"{MERT_LAYER_INDEX}"
)

MERT feature extraction layer: 6
Internal layer index: 5


In [ ]:
def extract_mert_features(
    audio_batch
):
    """
    Extract mean-pooled 768-D MERT features
    from Transformer block 6.
    """

    global captured_layer_output

    captured_layer_output = None

    # --------------------------------------------------------
    # Convert tensors to numpy
    # --------------------------------------------------------

    audio_numpy = [
        audio.cpu().numpy()
        for audio in audio_batch
    ]

    # --------------------------------------------------------
    # MERT processor
    # --------------------------------------------------------

    inputs = processor(
        audio_numpy,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # --------------------------------------------------------
    # MERT forward pass
    # --------------------------------------------------------

    with torch.inference_mode():

        outputs = model(
            **inputs,
            return_dict=True
        )

    # --------------------------------------------------------
    # Check that hook captured the layer
    # --------------------------------------------------------

    if captured_layer_output is None:

        raise RuntimeError(
            "MERT layer output was not captured."
        )

    hidden = captured_layer_output

    # --------------------------------------------------------
    # Expected shape:
    #
    # [batch_size, time_steps, 768]
    # --------------------------------------------------------

    if hidden.ndim != 3:

        raise RuntimeError(
            f"Unexpected MERT layer shape: "
            f"{hidden.shape}"
        )

    if hidden.shape[-1] != 768:

        raise RuntimeError(
            f"Expected hidden dimension 768, "
            f"got {hidden.shape[-1]}"
        )

    # --------------------------------------------------------
    # Temporal mean pooling
    #
    # [B, T, 768]
    #       ↓
    # [B, 768]
    # --------------------------------------------------------

    features = hidden.mean(
        dim=1
    )

    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    assert features.ndim == 2
    assert features.shape[-1] == 768

    return features

In [ ]:
# ============================================================
# STEP 9: EXTRACT MERT FEATURES AND SAVE TO CSV
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MERT_LAYER = 6          # Human-readable layer number: 6th encoder block
MERT_LAYER_INDEX = 5    # Python index: 0-based
BATCH_SIZE = 16

OUTPUT_CSV = os.path.join(
    path,
    "mert_layer6_features.csv"
)

print("Starting MERT feature extraction...")
print(f"Total files : {len(selected_files)}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"MERT layer  : {MERT_LAYER}")
print(f"Sample rate : {TARGET_SR} Hz")
print(f"Clip length : {CLIP_SECONDS} seconds")


# ------------------------------------------------------------
# Capture output from the 6th MERT encoder block
# ------------------------------------------------------------

captured_layer_output = None


def capture_layer_output(module, inputs, output):
    global captured_layer_output

    # Some Transformer layers return tuples.
    if isinstance(output, tuple):
        captured_layer_output = output[0]
    else:
        captured_layer_output = output


# Register hook on the 6th encoder block.
#
# IMPORTANT:
# model.encoder.layers[5] = 6th block
#
# because Python indexing starts at 0.
layer_handle = model.encoder.layers[MERT_LAYER_INDEX].register_forward_hook(
    capture_layer_output
)


# ------------------------------------------------------------
# MERT feature extraction function
# ------------------------------------------------------------

def extract_mert_features(audio_batch):
    """
    Extract the output of the 6th MERT encoder block.

    Input:
        audio_batch: list of torch tensors,
                    each containing one 3-second audio clip.

    Output:
        Tensor of shape [batch_size, 768]
    """

    global captured_layer_output

    captured_layer_output = None

    # Convert each audio tensor to numpy
    audio_numpy = [
        audio.cpu().numpy()
        for audio in audio_batch
    ]

    # Prepare input for MERT
    inputs = processor(
        audio_numpy,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True
    )

    # Move inputs to CPU/GPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Forward pass
    with torch.inference_mode():

        outputs = model(
            **inputs,
            return_dict=True
        )

    # --------------------------------------------------------
    # Check that our hook captured the desired layer
    # --------------------------------------------------------

    if captured_layer_output is None:
        raise RuntimeError(
            "MERT layer output was not captured."
        )

    hidden = captured_layer_output

    # Expected shape:
    # [batch_size, time_steps, 768]

    if hidden.ndim != 3:
        raise RuntimeError(
            f"Unexpected MERT layer shape: {hidden.shape}"
        )

    if hidden.shape[-1] != 768:
        raise RuntimeError(
            f"Expected hidden dimension 768, "
            f"got {hidden.shape[-1]}"
        )

    # --------------------------------------------------------
    # Mean-pool over the temporal dimension
    # --------------------------------------------------------

    features = hidden.mean(dim=1)

    # Expected:
    # [batch_size, 768]

    if features.ndim != 2:
        raise RuntimeError(
            f"Unexpected feature shape: {features.shape}"
        )

    if features.shape[-1] != 768:
        raise RuntimeError(
            f"Expected 768 features, "
            f"got {features.shape[-1]}"
        )

    return features


# ------------------------------------------------------------
# Test extraction on ONE file first
# ------------------------------------------------------------

print("\nTesting MERT extraction on one file...")

test_file = selected_files[0]["filepath"]

test_audio = load_and_prepare_audio(test_file)

test_features = extract_mert_features(
    [test_audio]
)

print("Test file:", test_file)
print("Feature shape:", test_features.shape)
print("First 10 features:")
print(test_features[0, :10])


# Expected:
# torch.Size([1, 768])

assert test_features.shape == (1, 768), \
    f"Unexpected test feature shape: {test_features.shape}"

print("\nTest passed.")


# ------------------------------------------------------------
# Extract features for all 14,000 files
# ------------------------------------------------------------

all_features = []
all_filepaths = []
all_labels = []


for start_idx in tqdm(
    range(0, len(selected_files), BATCH_SIZE),
    desc="Extracting MERT features"
):

    batch_items = selected_files[
        start_idx:start_idx + BATCH_SIZE
    ]

    batch_audio = []
    valid_items = []

    # --------------------------------------------------------
    # Load and prepare audio
    # --------------------------------------------------------

    for item in batch_items:

        try:

            audio = load_and_prepare_audio(
                item["filepath"]
            )

            batch_audio.append(audio)
            valid_items.append(item)

        except Exception as e:

            print(
                f"\nAudio loading error: "
                f"{item['filepath']}"
            )

            print(e)

    # Skip empty batches
    if len(batch_audio) == 0:
        continue

    # --------------------------------------------------------
    # Extract MERT features
    # --------------------------------------------------------

    try:

        batch_features = extract_mert_features(
            batch_audio
        )

    except Exception as e:

        print(
            f"\nMERT batch error "
            f"starting at index {start_idx}: {e}"
        )

        continue

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    batch_features = batch_features.cpu().numpy()

    all_features.append(batch_features)

    all_filepaths.extend(
        [item["filepath"] for item in valid_items]
    )

    all_labels.extend(
        [item["label"] for item in valid_items]
    )


# ------------------------------------------------------------
# Remove the forward hook
# ------------------------------------------------------------

layer_handle.remove()


# ------------------------------------------------------------
# Combine all features
# ------------------------------------------------------------

if len(all_features) == 0:
    raise RuntimeError(
        "No MERT features were extracted."
    )

all_features = np.concatenate(
    all_features,
    axis=0
)


# ------------------------------------------------------------
# Verify dimensions
# ------------------------------------------------------------

print("\nExtraction completed.")

print("Number of files :", len(all_filepaths))
print("Feature matrix  :", all_features.shape)
print("Number of labels:", len(all_labels))

assert all_features.shape[0] == len(all_filepaths)
assert all_features.shape[0] == len(all_labels)
assert all_features.shape[1] == 768


# ------------------------------------------------------------
# Create feature column names
# ------------------------------------------------------------

feature_columns = [
    f"mert_{i}"
    for i in range(768)
]


# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

features_df = pd.DataFrame(
    all_features,
    columns=feature_columns
)

features_df.insert(
    0,
    "label",
    all_labels
)

features_df.insert(
    0,
    "filepath",
    all_filepaths
)

features_df.insert(
    2,
    "mert_layer",
    MERT_LAYER
)


# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

features_df.to_csv(
    OUTPUT_CSV,
    index=False
)


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\n================================================")
print("MERT FEATURE EXTRACTION COMPLETE")
print("================================================")

print("Output file:")
print(OUTPUT_CSV)

print("\nDataFrame shape:")
print(features_df.shape)

print("\nColumns:")
print(features_df.columns[:10].tolist())
print("...")
print(features_df.columns[-5:].tolist())

print("\nClass distribution:")
print(features_df["label"].value_counts())

print("\nFirst 5 rows:")
print(features_df.iloc[:5, :8])

print("\nSaved successfully.")

Starting MERT feature extraction...
Total files : 14000
Batch size  : 16
MERT layer  : 6
Sample rate : 24000 Hz
Clip length : 3 seconds

Testing MERT extraction on one file...
Test file: /root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1/music_dataset/Accordion/1342.wav
Feature shape: torch.Size([1, 768])
First 10 features:
tensor([-0.0445, -0.1564,  0.1951,  0.0876, -0.2053,  0.1763, -0.0068, -0.0242,
        -0.1025,  0.0640], device='cuda:0')

Test passed.


Extracting MERT features:   0%|          | 0/875 [00:00<?, ?it/s]


Extraction completed.
Number of files : 14000
Feature matrix  : (14000, 768)
Number of labels: 14000

MERT FEATURE EXTRACTION COMPLETE
Output file:
/root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1/mert_layer6_features.csv

DataFrame shape:
(14000, 771)

Columns:
['filepath', 'label', 'mert_layer', 'mert_0', 'mert_1', 'mert_2', 'mert_3', 'mert_4', 'mert_5', 'mert_6']
...
['mert_763', 'mert_764', 'mert_765', 'mert_766', 'mert_767']

Class distribution:
label
Accordion          1000
Acoustic_Guitar    1000
Banjo              1000
Bass_Guitar        1000
Drum_set           1000
Electro_Guitar     1000
flute              1000
Harmonium          1000
Keyboard           1000
Mandolin           1000
Organ              1000
Shakers            1000
Trombone           1000
Horn               1000
Name: count, dtype: int64

First 5 rows:
                                            filepath      label  mert_layer  \
0  /root/.cache/kagglehub/dataset

In [ ]:
from google.colab import files
files.download('/root/.cache/kagglehub/datasets/abdulvahap/music-instrunment-sounds-for-classification/versions/1/mert_layer6_features.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# CLUSTERING EXPERIMENTS ON MERT LAYER-6 FEATURES
#
# 4 FEATURE CONDITIONS:
#   1. Non-normalized, No PCA
#   2. Non-normalized, PCA
#   3. Standardized, No PCA
#   4. Standardized, PCA
#
# 1 CLUSTERING ALGORITHM:
#   1. K-Means++
#
# 4 EVALUATION METRICS:
#   1. Silhouette Score
#   2. ARI
#   3. NMI
#   4. Clustering Accuracy
#
# ADDITIONAL:
#   - Class-wise K-Means Clustering Accuracy
# ============================================================


# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================

import os
import time
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    confusion_matrix
)

from scipy.optimize import linear_sum_assignment


# ============================================================
# STEP 2: LOAD THE SAVED MERT CSV
# ============================================================

CSV_PATH = os.path.join(
    path,
    "mert_layer6_features.csv"
)

df = pd.read_csv(CSV_PATH)

print("=" * 70)
print("MERT CSV LOADED")
print("=" * 70)

print("Shape:", df.shape)


# ============================================================
# STEP 3: VERIFY THE DATA
# ============================================================

EXPECTED_SAMPLES = 14000
EXPECTED_FEATURES = 768
EXPECTED_CLASSES = 14
EXPECTED_SAMPLES_PER_CLASS = 1000

assert len(df) == EXPECTED_SAMPLES, (
    f"Expected {EXPECTED_SAMPLES} samples, "
    f"found {len(df)}"
)

assert df["label"].nunique() == EXPECTED_CLASSES, (
    f"Expected {EXPECTED_CLASSES} classes, "
    f"found {df['label'].nunique()}"
)

class_counts = df["label"].value_counts()

assert all(
    class_counts == EXPECTED_SAMPLES_PER_CLASS
), (
    "Not all classes contain exactly "
    f"{EXPECTED_SAMPLES_PER_CLASS} samples."
)

# Get the 768 MERT features
feature_columns = [
    f"mert_{i}"
    for i in range(EXPECTED_FEATURES)
]

assert all(
    col in df.columns
    for col in feature_columns
), "Some MERT feature columns are missing."

print("\nData verification PASSED.")

print("Samples :", len(df))
print("Classes :", df["label"].nunique())
print("Features:", len(feature_columns))

print("\nClass counts:")
print(class_counts)


# ============================================================
# STEP 4: CREATE FEATURE MATRIX AND TRUE LABELS
# ============================================================

X = df[feature_columns].values.astype(np.float32)

y_text = df["label"].values

# Convert class names to integer labels
class_names = sorted(
    df["label"].unique()
)

label_to_int = {
    label: i
    for i, label in enumerate(class_names)
}

y = np.array([
    label_to_int[label]
    for label in y_text
])

print("\nFeature matrix:")
print("X shape:", X.shape)

print("\nClasses:")
for i, class_name in enumerate(class_names):
    print(i, "=", class_name)


# ============================================================
# STEP 5: CLUSTERING ACCURACY
# ============================================================

def clustering_accuracy(y_true, y_pred):
    """
    Calculate overall clustering accuracy using
    the Hungarian algorithm.

    Cluster IDs are arbitrary, so the Hungarian algorithm
    finds the optimal one-to-one mapping between predicted
    clusters and ground-truth classes.
    """

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    row_ind, col_ind = linear_sum_assignment(
        -cm
    )

    correct = cm[row_ind, col_ind].sum()

    accuracy = correct / len(y_true)

    return accuracy


def class_wise_clustering_accuracy(
    y_true,
    y_pred,
    class_names
):
    """
    Calculate class-wise clustering accuracy after
    applying the optimal Hungarian cluster-to-class mapping.

    Returns:
        DataFrame containing:
        - Class
        - Correct
        - Total
        - Accuracy
    """

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(class_names))
    )

    # Find optimal cluster-to-class mapping
    row_ind, col_ind = linear_sum_assignment(
        -cm
    )

    # Mapping:
    # predicted cluster -> true class
    cluster_to_class = {
        cluster: true_class
        for true_class, cluster in zip(
            row_ind,
            col_ind
        )
    }

    # Convert predicted cluster IDs to matched class IDs
    y_pred_matched = np.array([
        cluster_to_class.get(
            cluster,
            -1
        )
        for cluster in y_pred
    ])

    class_results = []

    for class_id, class_name in enumerate(class_names):

        mask = y_true == class_id

        total = int(mask.sum())

        correct = int(
            (y_pred_matched[mask] == class_id).sum()
        )

        accuracy = correct / total

        class_results.append({
            "Class": class_name,
            "Correct": correct,
            "Total": total,
            "Accuracy": accuracy
        })

    return pd.DataFrame(class_results)


# ============================================================
# STEP 6: EVALUATION FUNCTION
# ============================================================

def evaluate_clustering(
    X_features,
    y_true,
    predicted_labels
):
    """
    Calculate all four requested clustering metrics.
    """

    silhouette = silhouette_score(
        X_features,
        predicted_labels,
        metric="euclidean"
    )

    ari = adjusted_rand_score(
        y_true,
        predicted_labels
    )

    nmi = normalized_mutual_info_score(
        y_true,
        predicted_labels
    )

    accuracy = clustering_accuracy(
        y_true,
        predicted_labels
    )

    return {
        "Silhouette": silhouette,
        "ARI": ari,
        "NMI": nmi,
        "Clustering Accuracy": accuracy
    }


# ============================================================
# STEP 7: PREPARE THE FOUR FEATURE CONDITIONS
# ============================================================

print("\n" + "=" * 70)
print("PREPARING FEATURE CONDITIONS")
print("=" * 70)

feature_sets = {}


# ------------------------------------------------------------
# CONDITION 1:
# Non-normalized + No PCA
# ------------------------------------------------------------

feature_sets[
    "Non-normalized + No PCA"
] = X

print(
    "\n1. Non-normalized + No PCA:",
    X.shape
)


# ------------------------------------------------------------
# CONDITION 2:
# Non-normalized + PCA
# ------------------------------------------------------------

print("\nRunning PCA on non-normalized features...")

pca_non_normalized = PCA(
    n_components=0.95,
    svd_solver="full",
    random_state=42
)

X_non_normalized_pca = (
    pca_non_normalized.fit_transform(X)
)

feature_sets[
    "Non-normalized + PCA"
] = X_non_normalized_pca

print(
    "2. Non-normalized + PCA:",
    X_non_normalized_pca.shape
)

print(
    "Explained variance:",
    pca_non_normalized.explained_variance_ratio_.sum()
)


# ------------------------------------------------------------
# STANDARDIZATION
# ------------------------------------------------------------

print("\nStandardizing MERT features...")

scaler = StandardScaler()

X_normalized = scaler.fit_transform(X).astype(
    np.float32
)

print(
    "Standardized feature matrix:",
    X_normalized.shape
)


# ------------------------------------------------------------
# CONDITION 3:
# Normalized + No PCA
# ------------------------------------------------------------

feature_sets[
    "Normalized + No PCA"
] = X_normalized

print(
    "\n3. Normalized + No PCA:",
    X_normalized.shape
)


# ------------------------------------------------------------
# CONDITION 4:
# Normalized + PCA
# ------------------------------------------------------------

print("\nRunning PCA on normalized features...")

pca_normalized = PCA(
    n_components=0.95,
    svd_solver="full",
    random_state=42
)

X_normalized_pca = (
    pca_normalized.fit_transform(X_normalized)
)

feature_sets[
    "Normalized + PCA"
] = X_normalized_pca

print(
    "4. Normalized + PCA:",
    X_normalized_pca.shape
)

print(
    "Explained variance:",
    pca_normalized.explained_variance_ratio_.sum()
)


# ============================================================
# STEP 8: DISPLAY FEATURE DIMENSIONS
# ============================================================

print("\n" + "=" * 70)
print("FEATURE CONDITION SUMMARY")
print("=" * 70)

for name, features in feature_sets.items():

    print(
        f"{name:30s} -> {features.shape}"
    )


# ============================================================
# STEP 9: RUN K-MEANS++ CLUSTERING EXPERIMENTS
# ============================================================

N_CLUSTERS = EXPECTED_CLASSES
RANDOM_STATE = 42

results = []

# Store class-wise results for all four conditions
class_wise_results = []


for condition_name, X_condition in feature_sets.items():

    print("\n")
    print("=" * 70)
    print(
        f"FEATURE CONDITION: {condition_name}"
    )
    print("=" * 70)

    print(
        "Feature shape:",
        X_condition.shape
    )


    # ========================================================
    # K-MEANS++
    # ========================================================

    print("\nRunning K-Means++...")

    start_time = time.time()

    kmeans = KMeans(
        n_clusters=N_CLUSTERS,
        init="k-means++",
        n_init=20,
        max_iter=300,
        random_state=RANDOM_STATE,
        algorithm="lloyd"
    )

    kmeans_labels = kmeans.fit_predict(
        X_condition
    )

    kmeans_time = time.time() - start_time

    print(
        f"K-Means++ completed in "
        f"{kmeans_time:.2f} seconds."
    )


    # --------------------------------------------------------
    # Evaluate K-Means++
    # --------------------------------------------------------

    print("Calculating K-Means++ metrics...")

    kmeans_metrics = evaluate_clustering(
        X_condition,
        y,
        kmeans_labels
    )

    print(
        "Silhouette:",
        kmeans_metrics["Silhouette"]
    )

    print(
        "ARI:",
        kmeans_metrics["ARI"]
    )

    print(
        "NMI:",
        kmeans_metrics["NMI"]
    )

    print(
        "Clustering Accuracy:",
        kmeans_metrics["Clustering Accuracy"]
    )


    # --------------------------------------------------------
    # Class-wise K-Means++ Accuracy
    # --------------------------------------------------------

    class_acc_df = class_wise_clustering_accuracy(
        y,
        kmeans_labels,
        class_names
    )

    print("\nClass-wise K-Means++ Accuracy:")
    print("-" * 70)

    for _, row in class_acc_df.iterrows():

        print(
            f"{row['Class']:20s} "
            f"{int(row['Correct']):4d}/"
            f"{int(row['Total']):4d} "
            f"({row['Accuracy'] * 100:.2f}%)"
        )

    # Add feature condition to class-wise results
    class_acc_df.insert(
        0,
        "Feature Condition",
        condition_name
    )

    class_wise_results.append(
        class_acc_df
    )


    # --------------------------------------------------------
    # Store overall K-Means++ results
    # --------------------------------------------------------

    results.append({
        "Feature Condition": condition_name,
        "Clustering Algorithm": "K-Means++",
        "Feature Dimensions": X_condition.shape[1],
        "Silhouette": kmeans_metrics[
            "Silhouette"
        ],
        "ARI": kmeans_metrics[
            "ARI"
        ],
        "NMI": kmeans_metrics[
            "NMI"
        ],
        "Clustering Accuracy": kmeans_metrics[
            "Clustering Accuracy"
        ],
        "Time (seconds)": kmeans_time
    })


# ============================================================
# STEP 10: OVERALL RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results)

print("\n\n")
print("=" * 100)
print("FINAL K-MEANS++ CLUSTERING RESULTS")
print("=" * 100)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# STEP 11: SAVE OVERALL RESULTS
# ============================================================

RESULTS_CSV = os.path.join(
    path,
    "mert_layer6_kmeans_clustering_results.csv"
)

results_df.to_csv(
    RESULTS_CSV,
    index=False
)

print("\nOverall K-Means++ results saved to:")
print(RESULTS_CSV)


# ============================================================
# STEP 12: SAVE CLASS-WISE RESULTS
# ============================================================

class_wise_results_df = pd.concat(
    class_wise_results,
    ignore_index=True
)

CLASSWISE_RESULTS_CSV = os.path.join(
    path,
    "mert_layer6_kmeans_classwise_accuracy.csv"
)

class_wise_results_df.to_csv(
    CLASSWISE_RESULTS_CSV,
    index=False
)

print("\nClass-wise K-Means++ results saved to:")
print(CLASSWISE_RESULTS_CSV)


# ============================================================
# STEP 13: RESULTS BY FEATURE CONDITION
# ============================================================

print("\n" + "=" * 100)
print("K-MEANS++ RESULTS BY FEATURE CONDITION")
print("=" * 100)

for condition in feature_sets.keys():

    print("\n" + "-" * 100)
    print(condition)
    print("-" * 100)

    condition_results = results_df[
        results_df["Feature Condition"] == condition
    ]

    print(
        condition_results[
            [
                "Clustering Algorithm",
                "Feature Dimensions",
                "Silhouette",
                "ARI",
                "NMI",
                "Clustering Accuracy"
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )


# ============================================================
# STEP 14: DISPLAY CLASS-WISE RESULTS
# ============================================================

print("\n" + "=" * 100)
print("CLASS-WISE K-MEANS++ CLUSTERING ACCURACY")
print("=" * 100)

for condition in feature_sets.keys():

    print("\n" + "-" * 100)
    print(condition)
    print("-" * 100)

    condition_class_results = (
        class_wise_results_df[
            class_wise_results_df["Feature Condition"]
            == condition
        ]
    )

    display_df = condition_class_results.copy()

    display_df["Accuracy"] = (
        display_df["Accuracy"] * 100
    )

    display_df["Accuracy"] = display_df[
        "Accuracy"
    ].map(
        lambda x: f"{x:.2f}%"
    )

    print(
        display_df[
            [
                "Class",
                "Correct",
                "Total",
                "Accuracy"
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# EXPERIMENT COMPLETED
# ============================================================

print("\n" + "=" * 100)
print("EXPERIMENT COMPLETED")
print("=" * 100)

print("\nFiles saved:")
print("1.", RESULTS_CSV)
print("2.", CLASSWISE_RESULTS_CSV)

print("\nTotal samples:", len(df))
print("Number of classes:", len(class_names))
print("Clusters:", N_CLUSTERS)
print("Clustering algorithm: K-Means++")
print("Feature conditions:", len(feature_sets))


MERT CSV LOADED
Shape: (14000, 771)

Data verification PASSED.
Samples : 14000
Classes : 14
Features: 768

Class counts:
label
Accordion          1000
Acoustic_Guitar    1000
Banjo              1000
Bass_Guitar        1000
Drum_set           1000
Electro_Guitar     1000
flute              1000
Harmonium          1000
Keyboard           1000
Mandolin           1000
Organ              1000
Shakers            1000
Trombone           1000
Horn               1000
Name: count, dtype: int64

Feature matrix:
X shape: (14000, 768)

Classes:
0 = Accordion
1 = Acoustic_Guitar
2 = Banjo
3 = Bass_Guitar
4 = Drum_set
5 = Electro_Guitar
6 = Harmonium
7 = Horn
8 = Keyboard
9 = Mandolin
10 = Organ
11 = Shakers
12 = Trombone
13 = flute

PREPARING FEATURE CONDITIONS

1. Non-normalized + No PCA: (14000, 768)

Running PCA on non-normalized features...
2. Non-normalized + PCA: (14000, 108)
Explained variance: 0.95018053

Standardizing MERT features...
Standardized feature matrix: (14000, 768)

3. Normalized

In [ ]:
# ============================================================
# CLUSTERING EXPERIMENTS ON MERT LAYER-6 FEATURES
#
# 4 FEATURE CONDITIONS:
#   1. Raw, No PCA
#   2. Raw, PCA
#   3. L2-normalized, No PCA
#   4. L2-normalized, PCA
#
# 1 CLUSTERING ALGORITHM:
#   1. K-Means++
#
# 4 EVALUATION METRICS:
#   1. Silhouette Score
#   2. ARI
#   3. NMI
#   4. Clustering Accuracy
#
# ADDITIONAL:
#   - Class-wise K-Means++ Clustering Accuracy
#
# NORMALIZATION:
#   L2 normalization is applied independently to each
#   768-dimensional MERT embedding.
# ============================================================


# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================

import os
import time
import numpy as np
import pandas as pd

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    confusion_matrix
)

from scipy.optimize import linear_sum_assignment


# ============================================================
# STEP 2: LOAD THE SAVED MERT CSV
# ============================================================

CSV_PATH = os.path.join(
    path,
    "mert_layer6_features.csv"
)

df = pd.read_csv(CSV_PATH)

print("=" * 70)
print("MERT CSV LOADED")
print("=" * 70)

print("Shape:", df.shape)


# ============================================================
# STEP 3: VERIFY THE DATA
# ============================================================

EXPECTED_SAMPLES = 14000
EXPECTED_FEATURES = 768
EXPECTED_CLASSES = 14
EXPECTED_SAMPLES_PER_CLASS = 1000

assert len(df) == EXPECTED_SAMPLES, (
    f"Expected {EXPECTED_SAMPLES} samples, "
    f"found {len(df)}"
)

assert df["label"].nunique() == EXPECTED_CLASSES, (
    f"Expected {EXPECTED_CLASSES} classes, "
    f"found {df['label'].nunique()}"
)

class_counts = df["label"].value_counts()

assert all(
    class_counts == EXPECTED_SAMPLES_PER_CLASS
), (
    "Not all classes contain exactly "
    f"{EXPECTED_SAMPLES_PER_CLASS} samples."
)

# Get the 768 MERT features
feature_columns = [
    f"mert_{i}"
    for i in range(EXPECTED_FEATURES)
]

assert all(
    col in df.columns
    for col in feature_columns
), "Some MERT feature columns are missing."

print("\nData verification PASSED.")

print("Samples :", len(df))
print("Classes :", df["label"].nunique())
print("Features:", len(feature_columns))

print("\nClass counts:")
print(class_counts)


# ============================================================
# STEP 4: CREATE FEATURE MATRIX AND TRUE LABELS
# ============================================================

X = df[feature_columns].values.astype(np.float32)

y_text = df["label"].values

# Convert class names to integer labels
class_names = sorted(
    df["label"].unique()
)

label_to_int = {
    label: i
    for i, label in enumerate(class_names)
}

y = np.array([
    label_to_int[label]
    for label in y_text
])

print("\nFeature matrix:")
print("X shape:", X.shape)

print("\nClasses:")
for i, class_name in enumerate(class_names):
    print(i, "=", class_name)


# ============================================================
# STEP 5: CLUSTERING ACCURACY
# ============================================================

def clustering_accuracy(y_true, y_pred):
    """
    Calculate overall clustering accuracy using
    the Hungarian algorithm.

    Cluster IDs are arbitrary, so the Hungarian algorithm
    finds the optimal one-to-one mapping between predicted
    clusters and ground-truth classes.
    """

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    row_ind, col_ind = linear_sum_assignment(
        -cm
    )

    correct = cm[row_ind, col_ind].sum()

    accuracy = correct / len(y_true)

    return accuracy


def class_wise_clustering_accuracy(
    y_true,
    y_pred,
    class_names
):
    """
    Calculate class-wise clustering accuracy after
    applying the optimal Hungarian cluster-to-class mapping.

    Returns:
        DataFrame containing:
        - Class
        - Correct
        - Total
        - Accuracy
    """

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(class_names))
    )

    # Find optimal cluster-to-class mapping
    row_ind, col_ind = linear_sum_assignment(
        -cm
    )

    # Mapping:
    # predicted cluster -> true class
    cluster_to_class = {
        cluster: true_class
        for true_class, cluster in zip(
            row_ind,
            col_ind
        )
    }

    # Convert predicted cluster IDs to matched class IDs
    y_pred_matched = np.array([
        cluster_to_class.get(
            cluster,
            -1
        )
        for cluster in y_pred
    ])

    class_results = []

    for class_id, class_name in enumerate(class_names):

        mask = y_true == class_id

        total = int(mask.sum())

        correct = int(
            (y_pred_matched[mask] == class_id).sum()
        )

        accuracy = correct / total

        class_results.append({
            "Class": class_name,
            "Correct": correct,
            "Total": total,
            "Accuracy": accuracy
        })

    return pd.DataFrame(class_results)


# ============================================================
# STEP 6: EVALUATION FUNCTION
# ============================================================

def evaluate_clustering(
    X_features,
    y_true,
    predicted_labels
):
    """
    Calculate all four requested clustering metrics.
    """

    silhouette = silhouette_score(
        X_features,
        predicted_labels,
        metric="euclidean"
    )

    ari = adjusted_rand_score(
        y_true,
        predicted_labels
    )

    nmi = normalized_mutual_info_score(
        y_true,
        predicted_labels
    )

    accuracy = clustering_accuracy(
        y_true,
        predicted_labels
    )

    return {
        "Silhouette": silhouette,
        "ARI": ari,
        "NMI": nmi,
        "Clustering Accuracy": accuracy
    }


# ============================================================
# STEP 7: PREPARE THE FOUR FEATURE CONDITIONS
# ============================================================

print("\n" + "=" * 70)
print("PREPARING FEATURE CONDITIONS")
print("=" * 70)

feature_sets = {}


# ------------------------------------------------------------
# CONDITION 1:
# Raw + No PCA
# ------------------------------------------------------------

feature_sets[
    "Raw + No PCA"
] = X

print(
    "\n1. Raw + No PCA:",
    X.shape
)


# ------------------------------------------------------------
# CONDITION 2:
# Raw + PCA
# ------------------------------------------------------------

print("\nRunning PCA on raw features...")

pca_raw = PCA(
    n_components=0.95,
    svd_solver="full",
    random_state=42
)

X_raw_pca = (
    pca_raw.fit_transform(X)
)

feature_sets[
    "Raw + PCA"
] = X_raw_pca

print(
    "2. Raw + PCA:",
    X_raw_pca.shape
)

print(
    "Explained variance:",
    pca_raw.explained_variance_ratio_.sum()
)


# ------------------------------------------------------------
# L2 NORMALIZATION
# ------------------------------------------------------------

print("\nApplying L2 normalization...")

X_l2_normalized = normalize(
    X,
    norm="l2",
    axis=1
).astype(np.float32)

print(
    "L2-normalized feature matrix:",
    X_l2_normalized.shape
)

# Verify L2 normalization
l2_norms = np.linalg.norm(
    X_l2_normalized,
    axis=1
)

print(
    "Minimum L2 norm:",
    l2_norms.min()
)

print(
    "Maximum L2 norm:",
    l2_norms.max()
)


# ------------------------------------------------------------
# CONDITION 3:
# L2-normalized + No PCA
# ------------------------------------------------------------

feature_sets[
    "L2-normalized + No PCA"
] = X_l2_normalized

print(
    "\n3. L2-normalized + No PCA:",
    X_l2_normalized.shape
)


# ------------------------------------------------------------
# CONDITION 4:
# L2-normalized + PCA
# ------------------------------------------------------------

print(
    "\nRunning PCA on L2-normalized features..."
)

pca_l2 = PCA(
    n_components=0.95,
    svd_solver="full",
    random_state=42
)

X_l2_normalized_pca = (
    pca_l2.fit_transform(
        X_l2_normalized
    )
)

feature_sets[
    "L2-normalized + PCA"
] = X_l2_normalized_pca

print(
    "4. L2-normalized + PCA:",
    X_l2_normalized_pca.shape
)

print(
    "Explained variance:",
    pca_l2.explained_variance_ratio_.sum()
)


# ============================================================
# STEP 8: DISPLAY FEATURE DIMENSIONS
# ============================================================

print("\n" + "=" * 70)
print("FEATURE CONDITION SUMMARY")
print("=" * 70)

for name, features in feature_sets.items():

    print(
        f"{name:30s} -> {features.shape}"
    )


# ============================================================
# STEP 9: RUN K-MEANS++ CLUSTERING EXPERIMENTS
# ============================================================

N_CLUSTERS = EXPECTED_CLASSES
RANDOM_STATE = 42

results = []

# Store class-wise results for all four conditions
class_wise_results = []


for condition_name, X_condition in feature_sets.items():

    print("\n")
    print("=" * 70)
    print(
        f"FEATURE CONDITION: {condition_name}"
    )
    print("=" * 70)

    print(
        "Feature shape:",
        X_condition.shape
    )


    # ========================================================
    # K-MEANS++
    # ========================================================

    print("\nRunning K-Means++...")

    start_time = time.time()

    kmeans = KMeans(
        n_clusters=N_CLUSTERS,
        init="k-means++",
        n_init=20,
        max_iter=300,
        random_state=RANDOM_STATE,
        algorithm="lloyd"
    )

    kmeans_labels = kmeans.fit_predict(
        X_condition
    )

    kmeans_time = time.time() - start_time

    print(
        f"K-Means++ completed in "
        f"{kmeans_time:.2f} seconds."
    )


    # --------------------------------------------------------
    # Evaluate K-Means++
    # --------------------------------------------------------

    print("Calculating K-Means++ metrics...")

    kmeans_metrics = evaluate_clustering(
        X_condition,
        y,
        kmeans_labels
    )

    print(
        "Silhouette:",
        kmeans_metrics["Silhouette"]
    )

    print(
        "ARI:",
        kmeans_metrics["ARI"]
    )

    print(
        "NMI:",
        kmeans_metrics["NMI"]
    )

    print(
        "Clustering Accuracy:",
        kmeans_metrics["Clustering Accuracy"]
    )


    # --------------------------------------------------------
    # Class-wise K-Means++ Accuracy
    # --------------------------------------------------------

    class_acc_df = class_wise_clustering_accuracy(
        y,
        kmeans_labels,
        class_names
    )

    print("\nClass-wise K-Means++ Accuracy:")
    print("-" * 70)

    for _, row in class_acc_df.iterrows():

        print(
            f"{row['Class']:20s} "
            f"{int(row['Correct']):4d}/"
            f"{int(row['Total']):4d} "
            f"({row['Accuracy'] * 100:.2f}%)"
        )

    # Add feature condition
    class_acc_df.insert(
        0,
        "Feature Condition",
        condition_name
    )

    class_wise_results.append(
        class_acc_df
    )


    # --------------------------------------------------------
    # Store overall K-Means++ results
    # --------------------------------------------------------

    results.append({
        "Feature Condition": condition_name,
        "Clustering Algorithm": "K-Means++",
        "Feature Dimensions": X_condition.shape[1],
        "Silhouette": kmeans_metrics[
            "Silhouette"
        ],
        "ARI": kmeans_metrics[
            "ARI"
        ],
        "NMI": kmeans_metrics[
            "NMI"
        ],
        "Clustering Accuracy": kmeans_metrics[
            "Clustering Accuracy"
        ],
        "Time (seconds)": kmeans_time
    })


# ============================================================
# STEP 10: OVERALL RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results)

print("\n\n")
print("=" * 100)
print("FINAL K-MEANS++ CLUSTERING RESULTS")
print("=" * 100)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# STEP 11: SAVE OVERALL RESULTS
# ============================================================

RESULTS_CSV = os.path.join(
    path,
    "mert_layer6_kmeans_clustering_results.csv"
)

results_df.to_csv(
    RESULTS_CSV,
    index=False
)

print("\nOverall K-Means++ results saved to:")
print(RESULTS_CSV)


# ============================================================
# STEP 12: SAVE CLASS-WISE RESULTS
# ============================================================

class_wise_results_df = pd.concat(
    class_wise_results,
    ignore_index=True
)

CLASSWISE_RESULTS_CSV = os.path.join(
    path,
    "mert_layer6_kmeans_classwise_accuracy.csv"
)

class_wise_results_df.to_csv(
    CLASSWISE_RESULTS_CSV,
    index=False
)

print("\nClass-wise K-Means++ results saved to:")
print(CLASSWISE_RESULTS_CSV)


# ============================================================
# STEP 13: RESULTS BY FEATURE CONDITION
# ============================================================

print("\n" + "=" * 100)
print("K-MEANS++ RESULTS BY FEATURE CONDITION")
print("=" * 100)

for condition in feature_sets.keys():

    print("\n" + "-" * 100)
    print(condition)
    print("-" * 100)

    condition_results = results_df[
        results_df["Feature Condition"] == condition
    ]

    print(
        condition_results[
            [
                "Clustering Algorithm",
                "Feature Dimensions",
                "Silhouette",
                "ARI",
                "NMI",
                "Clustering Accuracy"
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )


# ============================================================
# STEP 14: DISPLAY CLASS-WISE RESULTS
# ============================================================

print("\n" + "=" * 100)
print("CLASS-WISE K-MEANS++ CLUSTERING ACCURACY")
print("=" * 100)

for condition in feature_sets.keys():

    print("\n" + "-" * 100)
    print(condition)
    print("-" * 100)

    condition_class_results = (
        class_wise_results_df[
            class_wise_results_df["Feature Condition"]
            == condition
        ]
    )

    display_df = condition_class_results.copy()

    display_df["Accuracy"] = (
        display_df["Accuracy"] * 100
    )

    display_df["Accuracy"] = display_df[
        "Accuracy"
    ].map(
        lambda x: f"{x:.2f}%"
    )

    print(
        display_df[
            [
                "Class",
                "Correct",
                "Total",
                "Accuracy"
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# EXPERIMENT COMPLETED
# ============================================================

print("\n" + "=" * 100)
print("EXPERIMENT COMPLETED")
print("=" * 100)

print("\nFiles saved:")
print("1.", RESULTS_CSV)
print("2.", CLASSWISE_RESULTS_CSV)

print("\nTotal samples:", len(df))
print("Number of classes:", len(class_names))
print("Clusters:", N_CLUSTERS)
print("Clustering algorithm: K-Means++")
print("Normalization: L2")
print("Feature conditions:", len(feature_sets))


MERT CSV LOADED
Shape: (14000, 771)

Data verification PASSED.
Samples : 14000
Classes : 14
Features: 768

Class counts:
label
Accordion          1000
Acoustic_Guitar    1000
Banjo              1000
Bass_Guitar        1000
Drum_set           1000
Electro_Guitar     1000
flute              1000
Harmonium          1000
Keyboard           1000
Mandolin           1000
Organ              1000
Shakers            1000
Trombone           1000
Horn               1000
Name: count, dtype: int64

Feature matrix:
X shape: (14000, 768)

Classes:
0 = Accordion
1 = Acoustic_Guitar
2 = Banjo
3 = Bass_Guitar
4 = Drum_set
5 = Electro_Guitar
6 = Harmonium
7 = Horn
8 = Keyboard
9 = Mandolin
10 = Organ
11 = Shakers
12 = Trombone
13 = flute

PREPARING FEATURE CONDITIONS

1. Raw + No PCA: (14000, 768)

Running PCA on raw features...
2. Raw + PCA: (14000, 108)
Explained variance: 0.95018053

Applying L2 normalization...
L2-normalized feature matrix: (14000, 768)
Minimum L2 norm: 0.9999987
Maximum L2 norm: 1.00